# 01 · EDA — flow traffic profile (Alert Triage #01)
Explore the merged NetFlow table: attack-class volumes, per-class feature fingerprints, and how separable the traffic is. Reusable logic lives in `src/evaluation/flow_traffic_profiler.py`; this notebook only explores.

In [ ]:
import sys
from pathlib import Path
SLOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(SLOT))
from src.config import load_all_configs
from src import data_source as ds
cfg = load_all_configs()
mcfg, fcfg = cfg['model'], cfg['feature']
selected = ds.load_selected(mcfg)['features']
print('dataset', mcfg['data']['dataset'], '| model features', selected)


## Load the table (all native columns)

In [ ]:
native = ds.native_features(mcfg)
df = ds.load_frame(mcfg, columns=native)
target, cls = mcfg['target'], mcfg['attack_class_column']
print(f'rows={len(df):,}  attack={df[target].mean()*100:.3f}%  native={len(native)}')

## Attack classes — every campaign in the capture

In [ ]:
vc = df[df[cls].str.upper() != 'BENIGN'][cls].value_counts()
vc.plot.barh(figsize=(9,6), color='#d03b3b', logx=True,
             title='Attack scenarios (count, log)');
print(vc.to_string())

## Column profile — counts, not percentages
`null_count` · `unique_count` · `duplicate_count` (distinct values appearing >1).

In [ ]:
from src.evaluation import flow_traffic_profiler as ftp
ftp.profile(df, selected + [target])

## Class-level share of traffic

In [ ]:
(df[cls].value_counts(normalize=True) * 100).round(3).head(20)